# Native multimodal decisions

Use a trained local checkpoint with the Python runtime. The example writes a synthetic image,
clip, waveform and JSONL records to an ignored run directory. Install the optional dependencies
from the source checkout with `python -m pip install -e '.[multimodal]'`.

The released Qwen checkpoint reads text, images and visual video. Set `CHECKPOINT` to a trained
audio-capable Gemma 4 checkpoint to try the waveform example. Download the released native
checkpoint into `models/jev-multimodal-qwen3.5-4b` using the linked quickstart.
[Installation and model download](../README.md#quickstart)

In [ ]:
from pathlib import Path

current = Path.cwd()
ROOT = next((p for p in (current, *current.parents) if (p / "laya" / "multimodal.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook from the repository checkout")
CHECKPOINT = ROOT / "models" / "jev-multimodal-qwen3.5-4b"
OUT = ROOT / "multimodal-runs" / "notebook-example"
OUT.mkdir(parents=True, exist_ok=True)

## Make synthetic media

The image contains a red square on a gray background. FFmpeg makes a short silent video from the
image when it is available. The audio file contains one second of a steady tone.

In [ ]:
import math
import shutil
import struct
import subprocess
import wave
from PIL import Image, ImageDraw

image_path = OUT / "scene.png"
image = Image.new("RGB", (128, 128), (224, 226, 228))
ImageDraw.Draw(image).rectangle((32, 32, 95, 95), fill=(220, 40, 40))
image.save(image_path)

video_path = None
if shutil.which("ffmpeg") and shutil.which("ffprobe"):
    video_path = OUT / "scene.mp4"
    subprocess.run(["ffmpeg", "-loglevel", "error", "-y", "-loop", "1", "-i", str(image_path),
                    "-t", "1", "-r", "8", "-an", "-c:v", "mpeg4", "-pix_fmt", "yuv420p", str(video_path)],
                   check=True)
else:
    print("Install FFmpeg and ffprobe to generate the video example")

audio_path = OUT / "tone.wav"
rate = 16_000
samples = [int(10000 * math.sin(2 * math.pi * 440 * t / rate)) for t in range(rate)]
with wave.open(str(audio_path), "wb") as audio:
    audio.setparams((1, 2, rate, 0, "NONE", "not compressed"))
    audio.writeframes(struct.pack(f"<{len(samples)}h", *samples))
print(OUT)

## Predict with the trained model

Text uses a choice question, the image tests a single referred box, and the clip uses a yes/no
decision. For eligible Gemma checkpoints, the final example asks directly about the audio signal.
The `usage` response includes model input tokens and forward passes.

In [ ]:
from laya_multimodal import MultimodalAgent

route_questions = {
    "route": {"type": "choice", "instructions": "Which team handles this request?",
              "criteria": {"billing": "invoices and refunds", "technical": "software issues"}},
}
box_questions = {"target": {"type": "box", "instructions": "the red square"}}
video_questions = {"shown": {"type": "noul", "instructions": "Is a red square visible in the clip?"}}
audio_questions = {"tone": {"type": "noul", "instructions": "Does the audio contain a steady tone?"}}
requests = [
    ("text", "I was charged twice for one invoice.", route_questions),
    ("image", {"image": str(image_path)}, box_questions),
]
if video_path is not None:
    requests.append(("video", {"video": str(video_path), "video_audio": False}, video_questions))

if (CHECKPOINT / "multimodal_config.json").is_file():
    agent = MultimodalAgent(CHECKPOINT)
    if agent.model_type == "gemma4":
        requests.append(("audio", {"audio": str(audio_path)}, audio_questions))
    for modality, state, questions in requests:
        result = agent.predict(state, questions)
        print(modality, result["answers"], result["usage"])
else:
    print("Set CHECKPOINT to the downloaded, trained native checkpoint to run inference.")

## Write Native JSONL examples

Decision rows store an expected option or probability target. A single-box row stores ordered
normalized corners `[left, top, right, bottom]`. The image and video share a group because they
show the same scene. These rows illustrate the format. Train, validation and test splits should
contain separate media groups. [Training, evaluation and data loading](../docs/multimodal.md)

In [ ]:
import json

rows = [
    {"id": "invoice", "group": "text-invoice", "state": "I was charged twice for one invoice.",
     "questions": route_questions, "expected": {"route": "billing"}},
    {"id": "square", "group": "square-scene", "state": {"image": str(image_path)},
     "questions": box_questions, "grounding": {"task": "single", "boxes": [[.25, .25, .75, .75]]}},
    {"id": "tone", "group": "audio-tone", "state": {"audio": str(audio_path)},
     "questions": audio_questions, "expected": {"tone": True}},
]
if video_path is not None:
    rows.append({"id": "square-clip", "group": "square-scene",
                 "state": {"video": str(video_path), "video_audio": False},
                 "questions": video_questions, "expected": {"shown": True}})
output = OUT / "example.jsonl"
output.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")
print(output, len(rows), "rows")